# Ứng dụng LightGBM dự đoán tiểu đường - Phiên bản tối ưu (No SMOTE + Calibration)

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, roc_auc_score

# Setup path
working_dir = Path.cwd().resolve()
repo_candidates = (working_dir, *working_dir.parents)
REPO_ROOT = next((path for path in repo_candidates if (path / "classification" / "lightgbm_classification.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Không tìm thấy repo root chứa classification/lightgbm_classification.py.")

repo_root_text = str(REPO_ROOT)
if repo_root_text not in sys.path:
    sys.path.insert(0, repo_root_text)

from classification.lightgbm_classification import LightGBMClassification
from classification.evaluation.run_diabetes_evaluation import (
    DEFAULT_OUTPUT_DIR,
    TARGET_COLUMN,
    build_diabetes_classifier,
    evaluate_diabetes_splits,
    split_diabetes_dataset,
    load_diabetes_dataset
)

In [ ]:
# Đọc dữ liệu
DATA_PATH = REPO_ROOT / "classification" / "data" / "diabetes_binary_health_indicators_BRFSS2015.csv"
df = pd.read_csv(DATA_PATH)
print('Kích thước dataset:', df.shape)
print('Phân bố nhãn:')
print(df[TARGET_COLUMN].value_counts())
print('Tỷ lệ lớp 1:', df[TARGET_COLUMN].mean())

X = df.drop(TARGET_COLUMN, axis=1)
y = df[TARGET_COLUMN]

In [ ]:
# Chia dữ liệu: Train 60% / Val 20% / Test 20% (Stratified)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val  # 0.25 * 0.8 = 0.2
)

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')
print(f'Train pos rate: {y_train.mean():.4f}')
print(f'Val pos rate: {y_val.mean():.4f}')
print(f'Test pos rate: {y_test.mean():.4f}')

In [ ]:
# Tính scale_pos_weight = số lượng lớp 0 / số lượng lớp 1 trên tập Train
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count
print(f'scale_pos_weight = {scale_pos_weight:.2f}')

# Reload module để lấy hyperparameter mới
import importlib
import classification.evaluation.run_diabetes_evaluation as diabetes_eval
diabetes_eval = importlib.reload(diabetes_eval)

# Khởi tạo model với scale_pos_weight
model = diabetes_eval.build_diabetes_classifier(scale_pos_weight=scale_pos_weight)
print('Model params:', {
    k: getattr(model, k) for k in [
        'n_estimators', 'learning_rate', 'num_leaves', 'max_depth',
        'min_child_samples', 'reg_alpha', 'reg_lambda', 'feature_fraction',
        'scale_pos_weight'
    ]
})

In [ ]:
# Training
model.fit(X_train, y_train)

# Dự đoán xác suất trên Validation (chưa hiệu chuẩn)
val_proba_raw = model.predict_proba(X_val)[:, 1]
test_proba_raw = model.predict_proba(X_test)[:, 1]

# ============================================================
# QUAN TRỌNG: HIỆU CHUẨN XÁC SUẤT (CALIBRATION)
# Custom LightGBM thường ra xác suất không hiệu chuẩn -> dùng Isotonic Regression
# ============================================================
calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(val_proba_raw, y_val)

val_proba_cal = calibrator.transform(val_proba_raw)
test_proba_cal = calibrator.transform(test_proba_raw)

print('Validation - Raw AUC:', roc_auc_score(y_val, val_proba_raw))
print('Validation - Calibrated AUC:', roc_auc_score(y_val, val_proba_cal))

In [ ]:
# ============================================================
# TÌM NGƯỠNG TỐI ƯU TRÊN VALIDATION (ĐÃ HIỆU CHUẨN)
# Mục tiêu: Max F1-score (không ràng buộc Accuracy)
# ============================================================
thresholds = np.arange(0.01, 0.99, 0.005)
best_f1 = 0
best_thresh = 0.5
best_metrics = {}

results = []
for th in thresholds:
    pred = (val_proba_cal >= th).astype(int)
    f1 = f1_score(y_val, pred, zero_division=0)
    prec = precision_score(y_val, pred, zero_division=0)
    rec = recall_score(y_val, pred, zero_division=0)
    acc = accuracy_score(y_val, pred)
    results.append({'thresh': th, 'f1': f1, 'prec': prec, 'rec': rec, 'acc': acc})
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = th
        best_metrics = {'f1': f1, 'prec': prec, 'rec': rec, 'acc': acc}

df_thresh = pd.DataFrame(results)
print('=== TOP 5 NGƯỠNG THEO F1 ===')
print(df_thresh.nlargest(5, 'f1').to_string(index=False))
print(f'\n>>> NGƯỠNG TỐI ƯU: {best_thresh:.3f}')
print(f'>>> Validation Metrics: F1={best_metrics["f1"]:.4f}, Prec={best_metrics["prec"]:.4f}, Rec={best_metrics["rec"]:.4f}, Acc={best_metrics["acc"]:.4f}')

# Set threshold cho model (dùng cho predict() nếu cần)
model.threshold = best_thresh

In [ ]:
# ============================================================
# ĐÁNH GIÁ CUỐI CÙNG TRÊN TEST SET (DÙNG XÁC SUẤT ĐÃ HIỆU CHUẨN)
# ============================================================
test_pred = (test_proba_cal >= best_thresh).astype(int)

test_acc = accuracy_score(y_test, test_pred)
test_prec = precision_score(y_test, test_pred, zero_division=0)
test_rec = recall_score(y_test, test_pred, zero_division=0)
test_f1 = f1_score(y_test, test_pred, zero_division=0)
test_auc = roc_auc_score(y_test, test_proba_cal)

print('='*50)
print('KẾT QUẢ CUỐI CÙNG TRÊN TEST SET')
print('='*50)
print(f'Accuracy : {test_acc:.4f}')
print(f'Precision: {test_prec:.4f}')
print(f'Recall   : {test_rec:.4f}')
print(f'F1-Score : {test_f1:.4f}')
print(f'ROC-AUC  : {test_auc:.4f}')
print('='*50)

# So sánh Train/Val/Test (dùng xác suất calibrated cho Val/Test)
train_proba_cal = calibrator.transform(model.predict_proba(X_train)[:, 1])
train_pred = (train_proba_cal >= best_thresh).astype(int)

metrics_comparison = pd.DataFrame({
    'Train': [
        accuracy_score(y_train, train_pred),
        precision_score(y_train, train_pred, zero_division=0),
        recall_score(y_train, train_pred, zero_division=0),
        f1_score(y_train, train_pred, zero_division=0),
        roc_auc_score(y_train, train_proba_cal)
    ],
    'Validation': [
        best_metrics['acc'],
        best_metrics['prec'],
        best_metrics['rec'],
        best_metrics['f1'],
        roc_auc_score(y_val, val_proba_cal)
    ],
    'Test': [
        test_acc,
        test_prec,
        test_rec,
        test_f1,
        test_auc
    ]
}, index=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'])

print(metrics_comparison.round(4))

In [ ]:
# ============================================================
# CHẠY ĐÁNH GIÁ CHUẨN CỦA REPO (TUYÊN TRUYỀN)
# Dùng evaluate_diabetes_splits để xuất artifact báo cáo
# Lưu ý: Hàm này dùng model.predict() với threshold đã set ở trên
# ============================================================
print('\nChạy pipeline đánh giá chuẩn repo...')
evaluation_run = evaluate_diabetes_splits(
    model=model,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    output_dir=DEFAULT_OUTPUT_DIR,
)

print('Đã lưu artifacts tại:', DEFAULT_OUTPUT_DIR)
print('Test Classification Report:')
print(evaluation_run['split_results']['test']['classification_report'])